In [1]:
import glob, os, json, shutil, time
import numpy as np, pandas as pd
from numpy.lib.format import open_memmap
from concurrent.futures import ThreadPoolExecutor

PART = 3          # third train subset (different samples from parts 1 and 2)
K_TRAIN = 17

PL = glob.glob("/kaggle/input/**/v2_places.csv", recursive=True)[0]
IN = os.path.dirname(PL); CONSDIR = os.path.join(IN, "cons")
pl = pd.read_csv(PL)
dirs = {int(os.path.basename(os.path.dirname(p))): os.path.dirname(p)
        for p in glob.glob("/kaggle/input/**/labels.npz", recursive=True)}
print("part:", PART, "| places:", len(pl), "| dirs:", len(dirs), "| cons dir:", CONSDIR)
print("disk /tmp free GB:", round(shutil.disk_usage("/tmp").free / 1e9, 1), "| /kaggle/working free GB:", round(shutil.disk_usage("/kaggle/working").free / 1e9, 1))
OUT = "/kaggle/working"

# samples already used by parts 1 and 2 (excluded here)
used = {}
for f in glob.glob("/kaggle/input/**/meta_train*.npz", recursive=True):
    m = np.load(f); v = m["valid"]
    print("exclusion file:", f, "| samples:", int(v.sum()))
    for a, b, c in zip(m["pid"][v], m["pp"][v], m["tt"][v]):
        used.setdefault(int(a), set()).add((int(b), int(c)))
print("earlier samples excluded:", sum(len(s) for s in used.values()))

def lut(d):
    t = np.full(256, 255, np.uint8)
    for k, v in d.items(): t[k] = v
    return t
WC = lut({10:1, 20:5, 30:2, 40:4, 50:6, 60:7, 70:8, 80:0, 90:3, 95:1, 100:7})
DW = lut({i: i for i in range(9)})
CLS = ["water", "trees", "grass", "flooded", "crops", "shrub", "built", "bare", "snow"]

def build(split, K, sfx=""):
    ids = sorted(pl[(pl.split == split) & (pl.months_ok >= 3)].id.tolist())
    N = len(ids) * K
    X = open_memmap(f"{OUT}/X_{split}{sfx}.npy", mode="w+", dtype=np.float16, shape=(N, 6, 256, 256))
    Y = open_memmap(f"{OUT}/Y_{split}{sfx}.npy", mode="w+", dtype=np.uint8, shape=(N, 3, 256, 256))
    V = np.zeros(N, bool); PID = np.zeros(N, np.int32); PP = np.zeros(N, np.int16); TT = np.zeros(N, np.int16)

    def work(a):
        si, pid = a
        rng = np.random.default_rng(pid + 1000 * PART)
        d = dirs[pid]
        months = {}
        for f in glob.glob(os.path.join(d, "20??-??.npz")):
            b = os.path.basename(f)[:-4]
            months[(int(b[:4]) - 2020) * 12 + int(b[5:7]) - 1] = f
        centers = [t for t in months if (t - 1) in months and (t + 1) in months]
        if not centers: return
        y = np.load(os.path.join(d, "labels.npz"))["y"]
        cons = np.load(f"{CONSDIR}/{pid:04d}.npz")["cons"]
        n = y.shape[0]
        ign = (cons == 255).mean((2, 3))
        ex_ = used.get(pid, ())
        cand = [(t, p) for t in centers for p in range(n)
                if ign[p, 0 if t < 12 else 1] < 0.6 and (p, t) not in ex_]
        if not cand: return
        sel = rng.choice(len(cand), size=min(K, len(cand)), replace=False)
        cache = {}
        def get(t):
            if t not in cache: cache[t] = np.load(months[t])["x"]
            return cache[t]
        for j, c in enumerate(sel):
            t, p = cand[c]; yi = 0 if t < 12 else 1
            x = np.concatenate([get(t - 1)[p], get(t)[p], get(t + 1)[p]]).astype(np.float32)
            x = np.clip(np.nan_to_num(x, nan=-35, neginf=-35, posinf=5), -35, 5)
            s = si * K + j
            X[s] = x.astype(np.float16)
            Y[s, 0] = cons[p, yi]
            Y[s, 1] = WC[y[p, yi]]
            Y[s, 2] = DW[y[p, 4 + yi]]
            V[s] = True; PID[s] = pid; PP[s] = p; TT[s] = t

    t0 = time.time()
    with ThreadPoolExecutor(4) as ex:
        for i, _ in enumerate(ex.map(work, enumerate(ids))):
            if i % 100 == 0: print(split, i, "/", len(ids), f"{time.time() - t0:.0f}s", flush=True)
    X.flush(); Y.flush()
    np.savez(f"{OUT}/meta_{split}{sfx}.npz", valid=V, pid=PID, pp=PP, tt=TT)

    vi = np.where(V)[0]
    print(f"\n{split}: valid samples {len(vi)} of {N} | places {len(set(PID[vi].tolist()))} of {len(ids)}")
    cnt = np.zeros((3, 256))
    for s in range(0, len(vi), 500):
        yy = Y[vi[s:s + 500]]
        for k in range(3): cnt[k] += np.bincount(yy[:, k].ravel(), minlength=256)
    json.dump(dict(consensus=cnt[0].tolist(), worldcover=cnt[1].tolist(), dynamicworld=cnt[2].tolist()),
              open(f"{OUT}/class_counts_{split}{sfx}.json", "w"))
    tab = {}
    for k, nm in enumerate(["consensus", "worldcover", "dynworld"]):
        c = cnt[k]; kn = c[:9].sum()
        tab[nm] = [100 * c[q] / kn for q in range(9)] + [100 * c[255] / c.sum()]
    print(pd.DataFrame(tab, index=CLS + ["ignored % of all"]).round(1).to_string())
    idx = np.sort(np.random.default_rng(0).choice(vi, min(100, len(vi)), replace=False))
    xs = X[idx].astype(np.float32)
    print("channel means (dB):", xs.mean((0, 2, 3)).round(2), "| NaN:", int(np.isnan(xs).sum()), "\n")

build("train", K_TRAIN, sfx=f"_p{PART}")
print("files:", sorted(os.listdir(OUT)))
print("total GB:", round(sum(os.path.getsize(f"{OUT}/{f}") for f in os.listdir(OUT) if os.path.isfile(f"{OUT}/{f}")) / 1e9, 2))

part: 3 | places: 1124 | dirs: 1124 | cons dir: /kaggle/input/notebooks/vinayybg/v2-01-consensus/cons
disk /tmp free GB: 1038.2 | /kaggle/working free GB: 20.9
exclusion file: /kaggle/input/notebooks/vinayybg/v2-02b-subsets/meta_train_p2.npz | samples: 15086
exclusion file: /kaggle/input/notebooks/vinayybg/v2-02-subsets/meta_train.npz | samples: 15152
earlier samples excluded: 30238
train 0 / 893 2s
train 100 / 893 57s
train 200 / 893 113s
train 300 / 893 165s
train 400 / 893 219s
train 500 / 893 271s
train 600 / 893 323s
train 700 / 893 374s
train 800 / 893 427s

train: valid samples 15039 of 15181 | places 885 of 893
                  consensus  worldcover  dynworld
water                   5.8         5.2       6.1
trees                  37.5        35.3      32.2
grass                  14.9        21.3       4.0
flooded                 1.5         2.4       0.8
crops                   9.5         8.9      10.0
shrub                   9.2         7.6      12.5
built                  